# AI 辅助切片（AI-Assisted Chunking）

**用途**：教学演示 - 使用 AI 按语义边界智能切片

**核心概念**：
- 利用 AI 识别语义边界
- 按主题/内容自然分段
- 切片质量最高，但需要 API 调用

## 运行前检查

1. 已安装依赖：`pip install -r ../requirements.txt`
2. 已完成 02_sliding_window.py 的学习
3. AI 切片需要 LLM API（本例使用阿里云百炼 DashScope）
4. 请确保 .env 文件中配置了 `ALIYUN_API_KEY`

## 第一部分：理解 AI 辅助切片

### 什么是 AI 辅助切片？

**定义**：
AI 辅助切片 = 使用大语言模型识别语义边界
让 AI 帮你决定在哪里切分文档最合适

**生活化比喻**：AI 切片 = "请专家帮忙切蛋糕"

- 固定切片：用尺子量，不管蛋糕上的装饰
- 滑动窗口：固定重叠，还是机械切割
- AI 切片：专家看着蛋糕，在装饰之间的空隙下刀

```
┌────────────────────────────────────────────┐
│  [主题 1：AI 简介] │ [主题 2：机器学习] │ [主题 3：深度学习] │
│  ← AI 识别出这里是主题边界，从这里切 →      │
└────────────────────────────────────────────┘
```

**工作原理**：
1. 将文档发送给 LLM
2. LLM 分析内容，识别主题/语义边界
3. LLM 返回建议的切分点
4. 按切分点分割文档

**优缺点**：

✓ 优点：
- 切片语义完整，质量最高
- 适应不同文档结构
- 不需要手动调参

✗ 缺点：
- 需要调用 LLM API，有成本
- 速度较慢
- 结果可能有波动（非确定性）

In [3]:
import os
from dotenv import load_dotenv
load_dotenv()

True

## 示例 1: 模拟 AI 切片（无 API 时的演示）

In [11]:
def mock_ai_chunking(text, max_chunk_size=200):
    """
    模拟 AI 切片（不使用真实 API）

    用启发式规则模拟"语义边界"识别
    实际使用时请替换为真实的 LLM 调用

    参数:
        text: 输入文本
        max_chunk_size: 最大切片大小
    返回:
        切片列表
    """
    import re

    # 模拟：按段落分割（假设段落是语义边界）
    paragraphs = re.split(r'\n\s*\n', text)

    chunks = []
    current_chunk = ""

    for para in paragraphs:
        para = para.strip()
        if not para:
            continue

        # 如果当前段落 + 已有内容超过限制
        if len(current_chunk) + len(para) > max_chunk_size:
            # 保存当前切片
            if current_chunk:
                chunks.append(current_chunk)
            # 开始新切片
            current_chunk = para
        else:
            # 添加到当前切片
            current_chunk += "\n\n" + para if current_chunk else para

    # 添加最后一片
    if current_chunk:
        chunks.append(current_chunk)

    return chunks

In [2]:
def demo_mock_ai_chunking():
    """
    演示模拟 AI 切片
    """
    print("=" * 60)
    print("示例 1: 模拟 AI 切片（无 API 时的演示）")
    print("=" * 60)

    # 测试文本（多段落，有不同主题）
    text = """人工智能（AI）是模拟人类智能的计算机科学领域。它包括机器学习、深度学习、自然语言处理等多个分支。

机器学习是 AI 的核心技术之一。它通过训练数据让计算机自动学习规律，无需显式编程。常见的机器学习算法包括决策树、神经网络、支持向量机等。

深度学习是机器学习的子集，使用多层神经网络模拟人脑。它在图像识别、语音识别等领域取得了突破性进展。卷积神经网络（CNN）和循环神经网络（RNN）是两种经典架构。

自然语言处理（NLP）让计算机理解和生成人类语言。应用包括机器翻译、情感分析、智能客服等。近年来，大语言模型（LLM）在 NLP 领域引发了革命。

计算机视觉（CV）让计算机能够"看懂"图像和视频。应用包括人脸识别、物体检测、医学图像分析等。深度学习大幅提升了 CV 的性能。"""

    print(f"原始文本：{len(text)} 字符")
    print(f"最大切片大小：200 字符\n")

    # 执行切片
    chunks = mock_ai_chunking(text, max_chunk_size=200)

    # 显示结果
    print(f"切片数量：{len(chunks)}\n")

    for i, chunk in enumerate(chunks):
        print(f"【语义切片 {i+1}】({len(chunk)} 字符)")
        print(f"  主题：{chunk[:30]}...")
        print(f"  内容预览：{chunk[:80]}...")
        print()


demo_mock_ai_chunking()

示例 1: 模拟 AI 切片（无 API 时的演示）
原始文本：343 字符
最大切片大小：200 字符

切片数量：2

【语义切片 1】(202 字符)
  主题：人工智能（AI）是模拟人类智能的计算机科学领域。它包括机器学...
  内容预览：人工智能（AI）是模拟人类智能的计算机科学领域。它包括机器学习、深度学习、自然语言处理等多个分支。

机器学习是 AI 的核心技术之一。它通过训练数据让计算机自...

【语义切片 2】(139 字符)
  主题：自然语言处理（NLP）让计算机理解和生成人类语言。应用包括机...
  内容预览：自然语言处理（NLP）让计算机理解和生成人类语言。应用包括机器翻译、情感分析、智能客服等。近年来，大语言模型（LLM）在 NLP 领域引发了革命。

计算机视觉...



## 示例 2: 使用 LLM 进行语义切片（真实 AI 切片 - 阿里云百炼 DashScope）

In [1]:
def ai_chunking_with_llm(text, max_chunks=5, api_key=None):
    """
    使用阿里云百炼 DashScope API 进行语义切片

    参数:
        text: 输入文本
        max_chunks: 期望的最大切片数
        api_key: API Key，默认从环境变量 DASHSCOPE_API_KEY 或 ALIYUN_API_KEY 读取
    返回:
        切片列表
    """
    import os
    import json
    from openai import OpenAI

    # 获取 API Key
    if api_key is None:
        api_key = os.getenv("DASHSCOPE_API_KEY") or os.getenv("ALIYUN_API_KEY")
        if not api_key:
            raise ValueError("未找到 API Key，请设置环境变量 DASHSCOPE_API_KEY 或 ALIYUN_API_KEY")

    # 初始化客户端（使用阿里云百炼兼容模式）
    client = OpenAI(
        api_key=api_key,
        base_url="https://dashscope.aliyuncs.com/compatible-mode/v1",
    )

    prompt = f"""请分析以下文本，找出{max_chunks}个语义边界（最适合切分的位置）。

文本：
========正文========
第一回 宴桃园豪杰三结义,斩黄巾英雄首立功
<p> 话说天下大势，分久必合，合久必分。周末七国分争，并入于秦。及秦灭之后，楚、汉分争，又并入于汉。汉朝自高祖斩白蛇而起义，一统天下;后来光武中兴，传至献帝，遂分为三国。推其致乱之由，殆始于桓、灵二帝。桓帝禁锢善类，崇信宦官。及桓帝崩，灵帝即位，大将军窦武、太傅陈蕃共相辅佐。时有宦官曹节等弄权，窦武、陈蕃谋诛之，机事不密，反为所害，中涓自此愈横。</p><p> 建宁二年四月望日，帝御温德殿。方升座，殿角狂风骤起。只见一条大青蛇，从梁上飞将下来，蟠于椅上。帝惊倒，左右急救入宫，百官俱奔避。须臾，蛇不见了。忽然大雷大雨，加以冰雹，落到半夜方止，坏却房屋无数。建宁四年二月，洛阳地震；又海水泛溢，沿海居民，尽被大浪卷入海中。光和元年，雌鸡化雄。六月朔，黑气十余丈，飞入温德殿中。秋七月，有虹现于玉堂；五原山岸，尽皆崩裂。种种不祥，非止一端。帝下诏问群臣以灾异之由，议郎蔡邕上疏，以为?堕鸡化，乃妇寺干政之所致，言颇切直。帝览奏叹息，因起更衣。曹节在后窃视，悉宣告左右；遂以他事陷邕于罪，放归田里。后张让、赵忠、封谞、段珪、曹节、侯览、蹇硕、程旷、夏恽、郭胜十人朋比为奸，号为"十常侍"。帝尊信张让，呼为"阿父"。朝政日非，以致天下人心思乱，盗贼蜂起。</p><p> 时巨鹿郡有兄弟三人，一名张角，一名张宝，一名张梁。那张角本是个不第秀才，因入山采药，遇一老人，碧眼童颜，手执藜杖，唤角至一洞中，以天书三卷授之，曰："此名《太平要术》，汝得之，当代天宣化，普救世人；若萌异心，必获恶报。"角拜问姓名。老人曰："吾乃南华老仙也。"言讫，化阵清风而去。角得此书，晓夜攻习，能呼风唤雨，号为"太平道人"。中平元年正月内，疫气流行，张角散施符水，为人治病，自称"大贤良师"。角有徒弟五百余人，云游四方，皆能书符念咒。次后徒众日多，角乃立三十六方，大方万余人，小方六七千，各立渠帅，称为将军；讹言："苍天已死，黄天当立；岁在甲子，天下大吉。"令人各以白土书"甲子"二字于家中大门上。青、幽、徐、冀、荆、扬、兖、豫八州之人，家家侍奉大贤良师张角名字。角遣其党马元义，暗赍金帛，结交中涓封谞，以为内应。角与二弟商议曰："至难得者，民心也。今民心已顺，若不乘势取天下，诚为可惜。"遂一面私造黄旗，约期举事；一面使弟子唐周，驰书报封谞。唐周乃径赴省中告变。帝召大将军何进调兵擒马元义，斩之；次收封谞等一干人下狱。张角闻知事露，星夜举兵，自称"天公将军"，张宝称"地公将军"，张梁称"人公将军"。申言于众曰："今汉运将终，大圣人出。汝等皆宜顺天从正，以乐太平。"四方百姓，裹黄巾从张角反者四五十万。贼势浩大，官军望风而靡。何进奏帝火速降诏，令各处备御，讨贼立功。一面遣中郎将卢植、皇甫嵩、朱儁，各引精兵、分三路讨之。</p><p> 且说张角一军，前犯幽州界分。幽州太守刘焉，乃江夏竟陵人氏，汉鲁恭王之后也。当时闻得贼兵将至，召校尉邹靖计议。靖曰："贼兵众，我兵寡，明公宜作速招军应敌。"刘焉然其说，随即出榜招募义兵。</p><p> 榜文行到涿县，引出涿县中一个英雄。那人不甚好读书；性宽和，寡言语，喜怒不形于色；素有大志，专好结交天下豪杰；生得身长七尺五寸，两耳垂肩，双手过膝，目能自顾其耳，面如冠玉，唇若涂脂；中山靖王刘胜之后，汉景帝阁下玄孙，姓刘名备，字玄德。昔刘胜之子刘贞，汉武时封涿鹿亭侯，后坐酎金失侯，因此遗这一枝在涿县。玄德祖刘雄，父刘弘。弘曾举孝廉，亦尝作吏，早丧。玄德幼孤，事母至孝；家贫，贩屦织席为业。家住本县楼桑村。其家之东南，有一大桑树，高五丈余，遥望之，童童如车盖。相者云："此家必出贵人。"玄德幼时，与乡中小儿戏于树下，曰："我为天子，当乘此车盖。"叔父刘元起奇其言，曰："此儿非常人也！"因见玄德家贫，常资给之。年十五岁，母使游学，尝师事郑玄、卢植，与公孙瓒等为友。及刘焉发榜招军时，玄德年已二十八岁矣。</p><p> 当日见了榜文，慨然长叹。随后一人厉声言曰："大丈夫不与国家出力，何故长叹？"玄德回视其人，身长八尺，豹头环眼，燕颔虎须，声若巨雷，势如奔马。玄德见他形貌异常，问其姓名。其人曰："某姓张名飞，字翼德。世居涿郡，颇有庄田，卖酒屠猪，专好结交天下豪杰。恰才见公看榜而叹，故此相问。"玄德曰："我本汉室宗亲，姓刘，名备。今闻黄巾倡乱，有志欲破贼安民，恨力不能，故长叹耳。"飞曰："吾颇有资财，当招募乡勇，与公同举大事，如何。"玄德甚喜，遂与同入村店中饮酒。</p><p> 正饮间，见一大汉，推着一辆车子，到店门首歇了，入店坐下，便唤酒保："快斟酒来吃，我待赶入城去投军。"玄德看其人：身长九尺，髯长二尺；面如重枣，唇若涂脂；丹凤眼，卧蚕眉，相貌堂堂，威风凛凛。玄德就邀他同坐，叩其姓名。其人曰："吾姓关名羽，字长生，后改云长，河东解良人也。因本处势豪倚势凌人，被吾杀了，逃难江湖，五六年矣。今闻此处招军破贼，特来应募。"玄德遂以己志告之，云长大喜。同到张飞庄上，共议大事。飞曰："吾庄后有一桃园，花开正盛；明日当于园中祭告天地，我三人结为兄弟，协力同心，然后可图大事。"玄德、云长齐声应曰："如此甚好。"</p><p> 次日，于桃园中，备下乌牛白马祭礼等项，三人焚香再拜而说誓曰："念刘备、关羽、张飞，虽然异姓，既结为兄弟，则同心协力，救困扶危；上报国家，下安黎庶。不求同年同月同日生，只愿同年同月同日死。皇天后土，实鉴此心，背义忘恩，天人共戮！"誓毕，拜玄德为兄，关羽次之，张飞为弟。祭罢天地，复宰牛设酒，聚乡中勇士，得三百余人，就桃园中痛饮一醉。来日收拾军器，但恨无马匹可乘。正思虑间，人报有两个客人，引一伙伴当，赶一群马，投庄上来。玄德曰："此天佑我也！"三人出庄迎接。原来二客乃中山大商：一名张世平，一名苏双，每年往北贩马，近因寇发而回。玄德请二人到庄，置酒管待，诉说欲讨贼安民之意。二客大喜，愿将良马五十匹相送；又赠金银五百两，镔铁一千斤，以资器用。</p><p> 玄德谢别二客，便命良匠打造双股剑。云长造青龙偃月刀，又名"冷艳锯"，重八十二斤。张飞造丈八点钢矛。各置全身铠甲。共聚乡勇五百余人，来见邹靖。邹靖引见太守刘焉。三人参见毕，各通姓名。玄德说起宗派，刘焉大喜，遂认玄德为侄。不数日，人报黄巾贼将程远志统兵五万来犯涿郡。刘焉令邹靖引玄德等三人，统兵五百，前去破敌。玄德等欣然领军前进，直至大兴山下，与贼相见。贼众皆披发，以黄巾抹额。当下两军相对，玄德出马，左有云长，右有翼德，扬鞭大骂："反国逆贼，何不早降！"程远志大怒，遣副将邓茂出战。张飞挺丈八蛇矛直出，手起处，刺中邓茂心窝，翻身落马。程远志见折了邓茂，拍马舞刀，直取张飞。云长舞动大刀，纵马飞迎。程远志见了，早吃一惊，措手不及，被云长刀起处，挥为两段。后人有诗赞二人曰：</p><p> 英雄露颖在今朝，<br/>一试矛兮一试刀。<br/>初出便将威力展，<br/>三分好把姓名标。</p><p> 众贼见程远志被斩，皆倒戈而走。玄德挥军追赶，投降者不计其数，大胜而回。刘焉亲自迎接，赏劳军士。次日，接得青州太守龚景牒文，言黄巾贼围城将陷，乞赐救援。刘焉与玄德商议。玄德曰："备愿往救之。"刘焉令邹靖将兵五千，同玄德、关、张，投青州来。贼众见救军至，分兵混战。玄德兵寡不胜，退三十里下寨。</p><p> 玄德谓关、张曰："贼众我寡；必出奇兵，方可取胜。"乃分关公引一千军伏山左，张飞引一千军伏山右，鸣金为号，齐出接应。次日，玄德与邹靖引军鼓噪而进。贼众迎战，玄德引军便退。贼众乘势追赶，方过山岭，玄德军中一齐鸣金，左右两军齐出，玄德摩军回身复杀。三路夹攻，贼众大溃。直赶至青州城下，太守龚景亦率民兵出城助战。贼势大败，剿戮极多，遂解青州之围。后人有诗赞玄德曰：运筹决算有神功，二虎还须逊一龙。初出便能垂伟绩，自应分鼎在孤穷。</p><p> 龚景犒军毕，邹靖欲回。玄德曰："近闻中郎将卢植与贼首张角战于广宗，备昔曾师事卢植，欲往助之。"于是邹靖引军自回，玄德与关、张引本部五百人投广宗来。至卢植军中，入帐施礼，具道来意。卢植大喜，留在帐前听调。</p><p> 时张角贼众十五万，植兵五万，相拒于广宗，未见胜负。植谓玄德曰："我今围贼在此，贼弟张梁、张宝在颍川，与皇甫嵩、朱儁对垒。汝可引本部人马，我更助汝一千官军，前去颍川打探消息，约期剿捕。"玄德领命，引军星夜投颍川来。</p><p> 时皇甫嵩、朱儁领军拒贼，贼战不利，退入长社，依草结营。嵩与□计曰："贼依草结营，当用火攻之。"遂令军士，每人束草一把，暗地埋伏。其夜大风忽起。二更以后，一齐纵火，嵩与□各引兵攻击贼寨，火焰张天，贼众惊慌，马不及鞍，人不及甲，四散奔走。</p><p> 杀到天明，张梁、张宝引败残军士，夺路而走。忽见一彪军马，尽打红旗，当头来到，截住去路。为首闪出一将，身长七尺，细眼长髯，官拜骑都尉，沛国谯郡人也，姓曹名操字孟德。操父曹嵩，本姓夏侯氏，因为中常侍曹腾之养子，故冒姓曹。曹嵩生操，小字阿瞒，一名吉利。操幼时，好游猎，喜歌舞，有权谋，多机变。操有叔父，见操游荡无度，尝怒之，言于曹嵩。嵩责操。操忽心生一计，见叔父来，诈倒于地，作中风之状。叔父惊告嵩，嵩急视之。操故无恙。嵩曰："叔言汝中风，今已愈乎？"操曰："儿自来无此病；因失爱于叔父，故见罔耳。"嵩信其言。后叔父但言操过，嵩并不听。因此，操得恣意放荡。时人有桥玄者，谓操曰："天下将乱，非命世之才不能济。能安之者，其在君乎？"南阳何?见操，言："汉室将亡，安天下者，必此人也。"汝南许劭，有知人之名。操往见之，问曰："我何如人？"劭不答。又问，劭曰："子治世之能臣，乱世之奸雄也。"操闻言大喜。年二十，举孝廉，为郎，除洛阳北部尉。初到任，即设五色棒十余条于县之四门，有犯禁者，不避豪贵，皆责之。中常侍蹇硕之叔，提刀夜行，操巡夜拿住，就棒责之。由是，内外莫敢犯者，威名颇震。后为顿丘令，因黄巾起，拜为骑都尉，引马步军五千，前来颍川助战。正值张梁、张宝败走，曹操拦住，大杀一阵，斩首万余级，夺得旗幡、金鼓、马匹极多。张梁、张宝死战得脱。操见过皇甫嵩、朱儁，随即引兵追袭张梁、张宝去了。</p><p> 却说玄德引关、张来颍川，听得喊杀之声，又望见火光烛天，急引兵来时，贼已败散。玄德见皇甫嵩、朱儁，具道卢植之意。嵩曰："张梁、张宝势穷力乏，必投广宗去依张角。玄德可即星夜往助。"玄德领命，遂引兵复回。到得半路，只见一簇军马，护送一辆槛车，车中之囚，乃卢植也。玄德大惊，滚鞍下马，问其缘故。植曰："我围张角，将次可破；因角用妖术，未能即胜。朝廷差黄门左丰前来体探，问我索取贿赂。我答曰：'军粮尚缺，安有余钱奉承天使？'左丰挟恨，回奏朝廷，说我高垒不战，惰慢军心；因此朝廷震怒，遣中郎将董卓来代将我兵，取我回京问罪。"张飞听罢，大怒，要斩护送军人，以救卢植。玄德急止之曰："朝廷自有公论，汝岂可造次？"军士簇拥卢植去了。关公曰："卢中郎已被逮，别人领兵，我等去无所依，不如且回涿郡。"玄德从其言，遂引军北行。行无二日，忽闻山后喊声大震。玄德引关、张纵马上高冈望之，见汉军大败，后面漫山塞野，黄巾盖地而来，旗上大书"天公将军"。玄德曰："此张角也！可速战！"三人飞马引军而出。张角正杀败董卓，乘势赴来，忽遇三人冲杀，角军大乱，败走五十余里。</p><p> 三人救了董卓回寨。卓问三人现居何职。玄德曰："白身。"卓甚轻之，不为礼。玄德出，张飞大怒曰："我等亲赴血战，救了这厮，他却如此无礼。若不杀之，难消我气！"便要提刀入帐来杀董卓。正是：人情势利古犹今，谁识英雄是白身？安得快人如翼德，尽诛世上负心人！毕竟董卓性命如何，且听下文分解。</p> 

第二回 张翼德怒鞭督邮,何国舅谋诛宦竖
且说董卓字仲颖，陇西临洮人也，官拜河东太守，自来骄傲。当日怠慢了玄德，张飞性发，便欲杀之。玄德与关公急止之曰；"他是朝廷命官，岂可擅杀？"飞曰："若不杀这厮，反要在他部下听令，其实不甘！二兄要便住在此，我自投别处去也！"玄德曰："我三人义同生死，岂可相离？不若都投别处去便了。"飞曰："若如此，稍解吾恨。"
于是三人连夜引军来投朱儁。□待之甚厚，合兵一处，进讨张宝。是时曹操自跟皇甫嵩讨张梁，大战于曲阳。这里朱儁进攻张宝。张宝引贼众八九万，屯于山后。□令玄德为其先锋，与贼对敌。张宝遣副将高升出马搦战，玄德使张飞击之。飞纵马挺矛，与升交战，不数合，刺升落马。玄德麾军直冲过去。张宝就马上披发仗剑，作起妖法。只见风雷大作，一股黑气从天而降，黑气中似有无限人马杀来。玄德连忙回军，军中大乱。败阵而归，与朱儁计议。□曰："彼用妖术，我来日可宰猪羊狗血，令军士伏于山头；候贼赶来，从高坡上泼之，其法可解。"玄德听令，拨关公、张飞各引军一千，伏于山后高冈之上，盛猪羊狗血并秽物准备。次日，张宝摇旗擂鼓，引军搦战，玄德出迎。交锋之际，张宝作法，风雷大作，飞砂走石，黑气漫天，滚滚人马，自天而下。玄德拨马便走，张宝驱兵赶来。将过山头，关、张伏军放起号炮，秽物齐泼。但见空中纸人草马，纷纷坠地；风雷顿息，砂石不飞。
张宝见解了法，急欲退军。左关公，右张飞，两军都出，背后玄德、朱儁一齐赶上，贼兵大败。玄德望见"地公将军"旗号，飞马赶来，张宝落荒而走。玄德发箭，中其左臂。张宝带箭逃脱，走入阳城，坚守不出。
朱儁引兵围住阳城攻打，一面差人打探皇甫嵩消息。探子回报，具说："皇甫嵩大获胜捷，朝廷以董卓屡败，命嵩代之。嵩到时，张角已死；张梁统其众，与我军相拒，被皇甫嵩连胜七阵，斩张梁于曲阳。发张角之棺，戮尸枭首，送往京师。余众俱降。朝廷加皇甫嵩为车骑将军，领冀州牧。皇甫嵩又表奏卢植有功无罪，朝廷复卢植原官。曹操亦以有功，除济南相，即日将班师赴任。"朱儁听说，催促军马，悉力攻打阳城。贼势危急，贼将严政刺杀张宝，献首投降。朱儁遂平数郡，上表献捷。时又黄巾余党三人：赵弘、韩忠、孙仲，聚众数万，望风烧劫，称与张角报仇。朝廷命朱儁即以得胜之师讨之。□奉诏，率军前进。时贼据宛城，□引兵攻之，赵弘遣韩忠出战。□遣玄德、关、张攻城西南角。韩忠尽率精锐之众，来西南角抵敌。朱儁自纵铁骑二千，径取东北角。贼恐失城，急弃西南面回。玄德从背后掩杀，贼众大败，奔入宛城。朱儁分兵四面围定。城中断粮，韩忠使人出城投降。□不许。玄德曰："昔高祖之得天下，盖为能招降纳顺；公何拒韩忠耶？"□曰："彼一时，此一时也。昔秦项之际，天下大乱，民无定主，故招降赏附，以劝来耳。今海内一统，惟黄巾造反；若容其降，无以劝善。使贼得利恣意劫掠，失利便投降：此长寇之志，非良策也。"玄德曰："不容寇降是矣。今四面围如铁桶，贼乞降不得，必然死战。万人一心，尚不可当，况城中有数万死命之人乎？不若撤去东南，独攻西北。贼必弃城而走，无心恋战，可即擒也。"□然之，随撤东南二面军马，一齐攻打西北。韩忠果引军弃城而奔。□与玄德、关、张率三军掩杀，射死韩忠，余皆四散奔走。正追赶间，赵弘、孙仲引贼众到，与□交战。□见弘势大，引军暂退。弘乘势复夺宛城。□离十里下寨。方欲攻打，忽见正东一彪人马到来。为首一将，生得广额阔面，虎体熊腰；吴郡富春人也，姓孙，名坚，字文台，乃孙武子之后。年十七岁时，与父至钱塘，见海贼十余人，劫取商人财物，于岸上分赃。坚谓父曰："此贼可擒也。"遂奋力提刀上岸，扬声大叫，东西指挥，如唤人状。贼以为官兵至，尽弃财物奔走。坚赶上，杀一贼。由是郡县知名，荐为校尉。后会稽妖贼许昌造反，自称"阳明皇帝"，聚众数万；坚与郡司马招募勇士千余人，会合州郡破之，斩许昌并其子许韶。刺史臧□上表奏其功，除坚为盐渎丞，又除盱眙丞、下邳丞。今见黄巾寇起，聚集乡中少年及诸商旅，并淮泗精兵一千五百余人，前来接应。
朱儁大喜，便令坚攻打南门，玄德打北门，朱儁打西门，留东门与贼走。孙坚首先登城，斩贼二十余人，贼众奔溃。赵弘飞马突槊，直取孙坚。坚从城上飞身夺弘槊，刺弘下马；却骑弘马，飞身往来杀贼。孙仲引贼突出北门，正迎玄德，无心恋战，只待奔逃。玄德张弓一箭，正中孙仲，翻身落马。朱儁大军随后掩杀，斩首数万级，降者不可胜计。南阳一路，十数郡皆平。□班师回京，诏封为车骑将军，河南尹。□表奏孙坚、刘备等功。坚有人情，除别郡司马上任去了。惟玄德听候日久，不得除授，三人郁郁不乐，上街闲行，正值郎中张钧车到。玄德见之，自陈功绩。钧大惊，随入朝见帝曰："昔黄巾造反，其原皆由十常侍卖官鬻爵，非亲不用，非仇不诛，以致天下大乱。今宜斩十常侍，悬首南郊，遣使者布告天下，有功者重加赏赐，则四海自清平也。"十常侍奏帝曰："张钧欺主。"帝令武士逐出张钧。十常侍共议："此必破黄巾有功者，不得除授，故生怨言。权且教省家铨注微名，待后却再理会未晚。"因此玄德除授定州中山府安喜县尉，克日赴任。
玄德将兵散回乡里，止带亲随二十余人，与关、张来安喜县中到任。署县事一月，与民秋毫无犯，民皆感化。到任之后，与关、张食则同桌，寝则同床。如玄德在稠人广坐，关、张侍立，终日不倦。到县未及四月，朝廷降诏，凡有军功为长吏者当沙汰。玄德疑在遣中。适督邮行部至县，玄德出郭迎接，见督邮施礼。督邮坐于马上，惟微以鞭指回答。关、张二公俱怒。及到馆驿，督邮南面高坐，玄德侍立阶下。良久，督邮问曰："刘县尉是何出身？"玄德曰："备乃中山靖王之后；自涿郡剿戮黄巾，大小三十余战，颇有微功，因得除今职。"督邮大喝曰："汝诈称皇亲，虚报功绩！目今朝廷降诏，正要沙汰这等滥官污吏！"玄德喏喏连声而退。归到县中，与县吏商议。吏曰："督邮作威，无非要贿赂耳。"玄德曰："我与民秋毫无犯，那得财物与他？"次日，督邮先提县吏去，勒令指称县尉害民。玄德几番自往求免，俱被门役阻住，不肯放参。
却说张飞饮了数杯闷酒，乘马从馆驿前过，见五六十个老人，皆在门前痛哭。飞问其故，众老人答曰："督邮逼勒县吏，欲害刘公；我等皆来苦告，不得放入，反遭把门人赶打！"张飞大怒，睁圆环眼，咬碎钢牙，滚鞍下马，径入馆驿，把门人那里阻挡得住，直奔后堂，见督邮正坐厅上，将县吏绑倒在地。飞大喝："害民贼！认得我么？"督邮未及开言，早被张飞揪住头发，扯出馆驿，直到县前马桩上缚住；攀下柳条，去督邮两腿上着力鞭打，一连打折柳条十数枝。玄德正纳闷间，听得县前喧闹，问左右，答曰："张将军绑一人在县前痛打。"玄德忙去观之，见绑缚者乃督邮也。玄德惊问其故，飞曰："此等害民贼，不打死等甚！"督邮告曰："玄德公救我性命！"玄德终是仁慈的人，急喝张飞住手。傍边转过关公来，曰："兄长建许多大功，仅得县尉，今反被督邮侮辱。吾思枳棘丛中，非栖鸾凤之所；不如杀督邮，弃官归乡，别图远大之计。"玄德乃取印绶，挂于督邮之颈，责之曰：据汝害民，本当杀却；今姑饶汝命。吾缴还印绶，从此去矣。"督邮归告定州太守，太守申文省府，差人捕捉。玄德、关、张三人往代州投刘恢。恢见玄德乃汉室宗亲，留匿在家不题。
却说十常侍既握重权，互相商议：但有不从己者，诛之。赵忠、张让差人问破黄巾将士索金帛，不从者奏罢职。皇甫嵩、朱儁皆不肯与，赵忠等俱奏罢其官。帝又封赵忠等为车骑将军，张让等十三人皆封列侯。朝政愈坏，人民嗟怨。于是长沙贼区星作乱；渔阳张举、张纯反：举称天子，纯称大将军。表章雪片告急，十常侍皆藏匿不奏。
一日，帝在后园与十常侍饮宴，谏议大夫刘陶，径到帝前大恸。帝问其故。陶曰："天下危在旦夕，陛下尚自与阉宦共饮耶！"帝曰："国家承平，有何危急？"陶曰："四方盗贼并起，侵掠州郡。其祸皆由十常侍卖官害民，欺君罔上。朝廷正人皆去，祸在目前矣！"十常侍皆免冠跪伏于帝前曰："大臣不相容，臣等不能活矣！愿乞性命归田里，尽将家产以助军资。"言罢痛哭。帝怒谓陶曰："汝家亦有近侍之人，何独不容朕耶？"呼武士推出斩之。刘陶大呼："臣死不惜！可怜汉室天下，四百余年，到此一旦休矣！"
武士拥陶出，方欲行刑，一大臣喝住曰："勿得下手，待我谏去。"众视之，乃司徒陈耽，径入宫中来谏帝曰："刘谏议得何罪而受诛？"帝曰："毁谤近臣，冒渎朕躬。"耽曰："天下人民，欲食十常侍之肉，陛下敬之如父母，身无寸功，皆封列侯；况封谞等结连黄巾，欲为内乱：陛下今不自省，社稷立见崩摧矣！"帝曰："封谞作乱，其事不明。十常侍中，岂无一二忠臣？"陈耽以头撞阶而谏。帝怒，命牵出，与刘陶皆下狱。是夜，十常侍即于狱中谋杀之；假帝诏以孙坚为长沙太守，讨区星，不五十日，报捷，江夏平，诏封坚为乌程侯。
封刘虞为幽州牧，领兵往渔阳征张举、张纯。代州刘恢以书荐玄德见虞。虞大喜，令玄德为都尉，引兵直抵贼巢，与贼大战数日，挫动锐气。张纯专一凶暴，士卒心变，帐下头目刺杀张纯，将头纳献，率众来降。张举见势败，亦自缢死。渔阳尽平。刘虞表奏刘备大功，朝廷赦免鞭督邮之罪，除下密丞，迁高堂尉。公孙瓒又表陈玄德前功，荐为别部司马，守平原县令。玄德在平原，颇有钱粮军马，重整旧日气象。刘虞平寇有功，封太尉。中平六年夏四月，灵帝病笃，召大将军何进入宫，商议后事。那何进起身屠家；因妹入宫为贵人，生皇子辩，遂立为皇后。进由是得权重任。帝又宠幸王美人，生皇子协。何后嫉妒，鸩杀王美人。皇子协养于董太后宫中。董太后乃灵帝之母，解渎亭侯刘苌之妻也。初因桓帝无子，迎立解渎亭侯之子，是为灵帝。灵帝入继大统，遂迎养母氏于宫中，尊为太后。董太后尝劝帝立皇子协为太子。帝亦偏爱协，欲立之。当时病笃，中常侍蹇硕奏曰："若欲立协，必先诛何进，以绝后患。"帝然其说，因宣进入宫。进至宫门，司马潘隐谓进曰："不可入宫。蹇硕欲谋杀公。"进大惊，急归私宅，召诸大臣，欲尽诛宦官。座上一人挺身出曰："宦官之势，起自冲、质之时；朝廷滋蔓极广，安能尽诛？倘机不密，必有灭族之祸：请细详之。"进视之，乃典军校尉曹操也。进叱曰："汝小辈安知朝廷大事！"正踌躇间，潘隐至，言："帝已崩。今赛硕与十常侍商议，秘不发丧，矫诏宣何国舅入宫，欲绝后患，册立皇子协为帝。"说未了，使命至，宣进速入，以定后事。操曰："今日之计，先宜正君位，然后图贼。"进曰："谁敢与吾正君讨贼？"一人挺身出曰："愿借精兵五千，斩关入内，册立新君，尽诛阉竖，扫清朝廷，以安天下！"进视之，乃司徒袁逢之子，袁隗之侄：名绍，字本初，现为司隶校尉。何进大喜，遂点御林军五千。绍全身披挂。何进引何?、荀攸、郑泰等大臣三十余员，相继而入，就灵帝柩前，扶立太子辩即皇帝位。
百官呼拜已毕，袁绍入宫收蹇硕。硕慌走入御园，花阴下为中常侍郭胜所杀。硕所领禁军，尽皆投顺。绍谓何进曰："中官结党。今日可乘势尽诛之。"张让等知事急，慌入告何后曰："始初设谋陷害大将军者，止赛硕一人，并不干臣等事。今大将军听袁绍之言，欲尽诛臣等，乞娘娘怜悯！"何太后曰："汝等勿忧，我当保汝。"传旨宣何进入。太后密谓曰："我与汝出身寒微，非张让等，焉能享此富贵？今蹇硕不仁，既已伏诛，汝何听信人言，欲尽诛宦官耶？"何进听罢，出谓众官曰："蹇硕设谋害我，可族灭其家。其余不必妄加残害。"袁绍曰："若不斩草除根，必为丧身之本。"进曰："吾意已决，汝勿多言。"众官皆退。次日，太后命何进参录尚书事，其余皆封官职。董太后宣张让等入宫商议曰："何进之妹，始初我抬举他。今日他孩儿即皇帝位，内外臣僚，皆其心腹：威权太重，我将如何？"让奏曰："娘娘可临朝，垂帘听政；封皇子协为王；加国舅董重大官，掌握军权；重用臣等：大事可图矣。"董太后大喜。次日设朝，董太后降旨，封皇子协为陈留王，董重为骠骑将军，张让等共预朝政。何太后见董太后专权，于宫中设一宴，请董太后赴席。酒至半酣，何太后起身捧杯再拜曰："我等皆妇人也，参预朝政，非其所宜。昔吕后因握重权，宗族千口皆被戮。今我等宜深居九重；朝廷大事，任大臣元老自行商议，此国家之幸也。愿垂听焉。"董后大怒曰："汝鸩死王美人，设心嫉妒。今倚汝子为君，与汝兄何进之势，辄敢乱言！吾敕骠骑断汝兄首，如反掌耳！"何后亦怒曰："吾以好言相劝，何反怒耶？"董后曰："汝家屠沽小辈，有何见识！"两宫互相争竞，张让等各劝归宫。何后连夜召何进入宫，告以前事。何进出，召三公共议。来早设朝，使廷臣奏董太后原系藩妃，不宜久居宫中，合仍迁于河间安置，限日下即出国门。一面遣人起送董后；一面点禁军围骠骑将军董重府宅，追索印绶。董重知事急，自刎于后堂。家人举哀，军士方散。张让、段珪见董后一枝已废，遂皆以金珠玩好结构何进弟何苗并其母舞阳君，令早晚入何太后处，善言遮蔽：因此十常侍又得近幸。
六月，何进暗使人鸩杀董后于河间驿庭，举柩回京，葬于文陵。进托病不出。司隶校尉袁绍入见进曰："张让、段珪等流言于外，言公鸩杀董后，欲谋大事。乘此时不诛阉宦，后必为大祸。昔窦武欲诛内竖，机谋不密，反受其殃。今公兄弟部曲将吏，皆英俊之士；若使尽力，事在掌握。此天赞之时，不可失也。"进曰："且容商议。"左右密报张让，让等转告何苗，又多送贿赂。苗入奏何后云："大将军辅佐新君，不行仁慈，专务杀伐。今无端又欲杀十常侍，此取乱之道也。"后纳其言。少顷，何进入白后，欲诛中涓。何后曰："中官统领禁省，汉家故事。先帝新弃天下，尔欲诛杀旧臣，非重宗庙也。"进本是没决断之人，听太后言，唯唯而出。袁绍迎问曰："大事若何？"进曰："太后不允，如之奈何？"绍曰："可召四方英雄之士，勒兵来京，尽诛阉竖。此时事急，不容太后不从。"进曰："此计大妙！"便发檄至各镇，召赴京师。主薄陈琳曰："不可！俗云：掩目而捕燕雀，是自欺也，微物尚不可欺以得志，况国家大事乎？今将军仗皇威，掌兵要，龙骧虎步，高下在心：若欲诛宦官，如鼓洪炉燎毛发耳。但当速发雷霆，行权立断，则天人顺之。却反外檄大臣，临犯京阙，英雄聚会，各怀一心：所谓倒持干戈，授人以柄，功必不成，反生乱矣。"何进笑曰："此懦夫之见也！"傍边一人鼓掌大笑曰："此事易如反掌，何必多议！"视之，乃曹操也。正是：欲除君侧宵人乱，须听朝中智士谋。不知曹操说出甚话来，且听下文分解。

请按以下 JSON 格式返回切分点（每个切分点是段落在文本中的起始索引位置）：
{{
    "split_points": [位置 1, 位置 2, ...],
    "reasons": ["为什么在这里切分的原因 1", "原因 2", ...]
}}

注意：
1. 切分点应该是段落或主题的边界
2. 保持每个切片的语义完整性
3. 返回纯 JSON，不要其他解释
4. 第一个切分点应该是 0
5. 最后一个切分点之后的内容应包含到文本末尾
"""

    print("=" * 60)
    print("AI 切片 - 调用阿里云百炼 API")
    print("=" * 60)
    print(f"\n调用模型：qwen-plus")
    print(f"文本长度：{len(text)} 字符")
    print("-" * 60)

    try:
        # 调用 API
        completion = client.chat.completions.create(
            model="qwen-plus",
            messages=[
                {"role": "system", "content": "You are a helpful assistant."},
                {"role": "user", "content": prompt},
            ]
        )

        # 解析响应
        response_text = completion.choices[0].message.content
        print(f"\nAPI 响应原始内容:\n{response_text[:500]}...")

        # 尝试解析 JSON
        # 清理可能的 markdown 标记
        if "```json" in response_text:
            response_text = response_text.split("```json")[1].split("```")[0].strip()
        elif "```" in response_text:
            response_text = response_text.split("```")[1].split("```")[0].strip()

        response = json.loads(response_text)
        split_points = response.get("split_points", [])
        reasons = response.get("reasons", [])

        print(f"\nLLM 返回的切分点:\n")
        print(f"  切分位置：{split_points}")
        print(f"  切分原因：{reasons[:3]}...")

        # 按切分点分割文本
        chunks = []
        if split_points:
            # 确保第一个点是 0
            if split_points[0] != 0:
                split_points.insert(0, 0)

            for i, start in enumerate(split_points):
                end = split_points[i + 1] if i + 1 < len(split_points) else len(text)
                chunk = text[start:end].strip()
                if chunk:
                    chunks.append(chunk)
        else:
            # 如果 API 没有返回有效的切分点，退化为按段落分割
            print("  警告：API 未返回有效切分点，退化为按段落分割")
            paragraphs = [p.strip() for p in text.split('\n\n') if p.strip()]
            chunks = paragraphs

        print(f"\n生成的切片数量：{len(chunks)}")
        return chunks

    except Exception as e:
        print(f"\nAPI 调用失败：{e}")
        print("退化为按段落分割...")
        # 退化方案：按段落分割
        paragraphs = [p.strip() for p in text.split('\n\n') if p.strip()]
        return paragraphs

In [4]:
def demo_real_ai_chunking():
    """
    演示真实 AI 切片（使用阿里云百炼 API）
    """
    print()
    print("=" * 60)
    print("示例 2: 真实 AI 切片（阿里云百炼 API）")
    print("=" * 60)

    # 测试文本
    text = """人工智能（AI）是模拟人类智能的计算机科学领域。它试图理解智能的本质，并生产出能以人类智能相似的方式做出反应的智能机器。该领域的研究包括机器人、语言识别、图像识别、自然语言处理和专家系统等。

机器学习是 AI 的核心技术之一，通过训练数据让计算机自动学习规律，无需显式编程。监督学习、无监督学习、强化学习是三种主要的学习范式。常见的算法包括决策树、随机森林、支持向量机、神经网络等。

深度学习是机器学习的子集，使用多层神经网络模拟人脑。卷积神经网络（CNN）在图像识别领域取得突破，循环神经网络（RNN）和 Transformer 架构在自然语言处理领域表现优异。深度学习的成功得益于大数据、强算力和算法创新。

自然语言处理（NLP）让计算机理解和生成人类语言。从早期的规则方法到统计方法，再到现在的深度学习方法，NLP 取得了长足进步。机器翻译、情感分析、文本生成、对话系统等应用已经深入人们的生活。

计算机视觉（CV）让计算机能够"看懂"图像和视频。目标检测、图像分割、人脸识别、姿态估计等任务都有了突破性进展。自动驾驶、医学影像分析、工业质检等应用正在改变各个行业。"""

    print(f"测试文本长度：{len(text)} 字符")
    print(f"期望切片数：3\n")

    # 调用 AI 切片
    chunks = ai_chunking_with_llm(text, max_chunks=3)

    print("\n" + "=" * 60)
    print("切片结果预览：")
    print("=" * 60)
    for i, chunk in enumerate(chunks):
        print(f"\n【切片 {i+1}】({len(chunk)} 字符)")
        print(f"  内容：{chunk[:80]}...")


# 运行示例 2（需要 API Key，如无请跳过）
demo_real_ai_chunking()


示例 2: 真实 AI 切片（阿里云百炼 API）
测试文本长度：490 字符
期望切片数：3

AI 切片 - 调用阿里云百炼 API

调用模型：qwen-plus
文本长度：490 字符
------------------------------------------------------------

API 响应原始内容:
```json
{
    "split_points": [0, 1024, 3768, 7295],
    "reasons": ["文本起始位置，第一回开篇，包含回目标题和开篇总论天下大势", "第一回核心情节转换点：从天下大势、灾异背景、黄巾起源，转入具体人物登场——刘关张三人正式出场并结义，语义重心由宏观历史叙述转向英雄叙事", "第一回高潮与收束处：桃园结义完成、初战告捷、解青州之围后，叙事节奏放缓；随后转入卢植被陷、董卓轻慢等新冲突，为第二回铺垫；此处是第一回主体事件（结义建功）的自然段落终点", "第二回开篇明确标识'第二回 张翼德怒鞭督邮,何国舅谋诛宦竖'，是全新回目的起始，主题转向官场权斗与个人气节，语义边界清晰，且与前文形成结构对称的章回体分界"]
}
```...

LLM 返回的切分点:

  切分位置：[0, 1024, 3768, 7295]
  切分原因：['文本起始位置，第一回开篇，包含回目标题和开篇总论天下大势', '第一回核心情节转换点：从天下大势、灾异背景、黄巾起源，转入具体人物登场——刘关张三人正式出场并结义，语义重心由宏观历史叙述转向英雄叙事', '第一回高潮与收束处：桃园结义完成、初战告捷、解青州之围后，叙事节奏放缓；随后转入卢植被陷、董卓轻慢等新冲突，为第二回铺垫；此处是第一回主体事件（结义建功）的自然段落终点']...

生成的切片数量：1

切片结果预览：

【切片 1】(490 字符)
  内容：人工智能（AI）是模拟人类智能的计算机科学领域。它试图理解智能的本质，并生产出能以人类智能相似的方式做出反应的智能机器。该领域的研究包括机器人、语言识别、图像识...


## 示例 3: 基于句子聚类的切片

In [5]:
def sentence_clustering_chunking(text, sentences_per_chunk=3):
    """
    基于句子聚类的切片

    这是一种简化版的"智能"切片：
    1. 先分割成句子
    2. 将相近的句子聚在一起
    3. 形成语义连贯的切片

    参数:
        text: 输入文本
        sentences_per_chunk: 每个切片的句子数
    返回:
        切片列表
    """
    import re

    # 1. 句子分割
    sentences = re.split(r'[.!?!.!?。！？]+', text)
    sentences = [s.strip() for s in sentences if s.strip()]

    if not sentences:
        return []

    # 2. 按句子分组（简化版聚类）
    chunks = []

    for i in range(0, len(sentences), sentences_per_chunk):
        group = sentences[i:i + sentences_per_chunk]
        chunk = "。".join(group) + "。"
        chunks.append(chunk)

    return chunks

In [6]:
def demo_sentence_clustering():
    """
    演示基于句子聚类的切片
    """
    print()
    print("=" * 60)
    print("示例 3: 基于句子聚类的切片")
    print("=" * 60)

    # 测试文本
    text = """人工智能是计算机科学的一个分支。它试图理解智能的本质。机器学习是 AI 的核心技术。
深度学习使用多层神经网络。它在图像识别领域取得突破。自然语言处理让计算机理解语言。
计算机视觉让机器看懂图像。推荐系统根据偏好推荐内容。知识图谱用图结构表示知识。
大语言模型基于海量文本训练。它能生成高质量的文本。AI 正在改变我们的生活。"""

    print(f"句子数量：估算...")
    print(f"每切片句子数：3\n")

    # 执行切片
    chunks = sentence_clustering_chunking(text, sentences_per_chunk=3)

    print(f"切片数量：{len(chunks)}\n")

    for i, chunk in enumerate(chunks):
        sentence_count = len([s for s in chunk.split('。') if s.strip()])
        print(f"【聚类切片 {i+1}】({sentence_count} 句子)")
        print(f"  {chunk}")
        print()


demo_sentence_clustering()


示例 3: 基于句子聚类的切片
句子数量：估算...
每切片句子数：3

切片数量：4

【聚类切片 1】(3 句子)
  人工智能是计算机科学的一个分支。它试图理解智能的本质。机器学习是 AI 的核心技术。

【聚类切片 2】(3 句子)
  深度学习使用多层神经网络。它在图像识别领域取得突破。自然语言处理让计算机理解语言。

【聚类切片 3】(3 句子)
  计算机视觉让机器看懂图像。推荐系统根据偏好推荐内容。知识图谱用图结构表示知识。

【聚类切片 4】(3 句子)
  大语言模型基于海量文本训练。它能生成高质量的文本。AI 正在改变我们的生活。



## 示例 4: 使用 LangChain 的文本分割器

In [11]:
# LangChain封装了一种递归的切片的方法

def langchain_text_splitter_demo():
    """
    演示使用 LangChain 的文本分割器

    LangChain 提供了多种智能文本分割器
    """
    print("=" * 60)
    print("示例 4: LangChain 文本分割器")
    print("=" * 60)

    try:
        from langchain_text_splitters import (
            CharacterTextSplitter,
            RecursiveCharacterTextSplitter,
            TokenTextSplitter
        )

        # 测试文本
        text = """人工智能（AI）是模拟人类智能的计算机科学领域。
它包括机器学习、深度学习、自然语言处理等多个分支。
机器学习通过训练数据让计算机自动学习规律。
深度学习使用多层神经网络模拟人脑。
自然语言处理让计算机理解和生成人类语言。
计算机视觉让计算机能够"看懂"图像和视频。"""

        print("1. CharacterTextSplitter（字符分割器）")
        print("-" * 40)

        char_splitter = CharacterTextSplitter(
            separator="\n",      # 按换行符分割
            chunk_size=100,      # 切片大小
            chunk_overlap=20,    # 重叠
            length_function=len  # 长度计算函数
        )

        char_chunks = char_splitter.split_text(text)
        print(f"切片数量：{len(char_chunks)}")
        for i, chunk in enumerate(char_chunks[:3]):
            print(f"  [{i+1}] {chunk[:50]}...")

        print()
        print("2. RecursiveCharacterTextSplitter（递归字符分割器）")
        print("-" * 40)

        recursive_splitter = RecursiveCharacterTextSplitter(
            separators=["\n\n", "\n", "。", ".", " "],  # 多级分隔符
            chunk_size=100,
            chunk_overlap=20,
            length_function=len
        )

        recursive_chunks = recursive_splitter.split_text(text)
        print(f"切片数量：{len(recursive_chunks)}")
        for i, chunk in enumerate(recursive_chunks[:3]):
            print(f"  [{i+1}] {chunk[:50]}...")

        print()
        print("3. TokenTextSplitter（Token 分割器）")
        print("-" * 40)
        print("  按 Token 数量分割（适合控制 LLM 输入）")
        print("  需要安装 tiktoken: pip install tiktoken")

    except ImportError as e:
        print(f"需要安装：pip install langchain-text-splitters")
        print(f"错误：{e}")


langchain_text_splitter_demo()

示例 4: LangChain 文本分割器


C:\沃林AI课程\AI0226_0309课程\wolin_learn-master\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


1. CharacterTextSplitter（字符分割器）
----------------------------------------
切片数量：2
  [1] 人工智能（AI）是模拟人类智能的计算机科学领域。
它包括机器学习、深度学习、自然语言处理等多个分支。...
  [2] 深度学习使用多层神经网络模拟人脑。
自然语言处理让计算机理解和生成人类语言。
计算机视觉让计算机能够...

2. RecursiveCharacterTextSplitter（递归字符分割器）
----------------------------------------
切片数量：2
  [1] 人工智能（AI）是模拟人类智能的计算机科学领域。
它包括机器学习、深度学习、自然语言处理等多个分支。...
  [2] 深度学习使用多层神经网络模拟人脑。
自然语言处理让计算机理解和生成人类语言。
计算机视觉让计算机能够...

3. TokenTextSplitter（Token 分割器）
----------------------------------------
  按 Token 数量分割（适合控制 LLM 输入）
  需要安装 tiktoken: pip install tiktoken


## 示例 5: AI 切片 vs 固定切片对比

In [12]:
def compare_chunking_methods(text):
    """
    对比 AI 切片和固定切片的效果
    """
    print("=" * 60)
    print("示例 5: AI 切片 vs 固定切片对比")
    print("=" * 60)

    # 1. 固定字符切片（本地定义避免导入问题）
    def fixed_char_chunking(t, chunk_size=150):
        return [t[i:i+chunk_size] for i in range(0, len(t), chunk_size)]

    fixed_chunks = fixed_char_chunking(text, chunk_size=150)

    # 2. 滑动窗口切片（本地定义）
    def sliding_window_chunking(t, window_size=150, step_size=75):
        chunks = []
        start = 0
        while start < len(t):
            chunks.append(t[start:start+window_size])
            start += step_size
            if start >= len(t):
                break
        return chunks

    sliding_chunks = sliding_window_chunking(text, window_size=150, step_size=75)

    # 3. 模拟 AI 切片
    ai_chunks = mock_ai_chunking(text, max_chunk_size=150)

    print(f"\n文本长度：{len(text)} 字符\n")

    print("┌────────────────┬──────────┬─────────────┬──────────────┐")
    print("│    切片方法     │ 切片数量  │  平均长度    │   语义完整度  │")
    print("├────────────────┼──────────┼─────────────┼──────────────┤")
    print(f"│  固定字符切片   │   {len(fixed_chunks):2d}     │  {sum(len(c) for c in fixed_chunks)/len(fixed_chunks):5.1f}   │     ⭐⭐        │")
    print(f"│  滑动窗口切片   │   {len(sliding_chunks):2d}     │  {sum(len(c) for c in sliding_chunks)/len(sliding_chunks):5.1f}   │     ⭐⭐⭐       │")
    print(f"│  AI 辅助切片     │   {len(ai_chunks):2d}     │  {sum(len(c) for c in ai_chunks)/len(ai_chunks):5.1f}   │     ⭐⭐⭐⭐      │")
    print("└────────────────┴──────────┴─────────────┴──────────────┘")

    print("\n💡 说明：")
    print("   语义完整度是主观评估，AI 切片通常能保持更好的语义连贯性")

In [13]:
def demo_method_comparison():
    """
    运行切片方法对比
    """
    # 测试文本（包含多个主题段落）
    text = """人工智能（AI）是模拟人类智能的计算机科学领域。它包括机器学习、深度学习、自然语言处理等多个分支。AI 从 1956 年诞生至今，已经经历了多次发展浪潮。

机器学习是 AI 的核心技术之一。它通过训练数据让计算机自动学习规律，无需显式编程。监督学习、无监督学习、强化学习是三种主要的学习范式。

深度学习是机器学习的子集，使用多层神经网络模拟人脑。它在图像识别、语音识别等领域取得了突破性进展。卷积神经网络（CNN）和循环神经网络（RNN）是两种经典架构。

自然语言处理（NLP）让计算机理解和生成人类语言。应用包括机器翻译、情感分析、智能客服等。近年来，大语言模型（LLM）在 NLP 领域引发了革命。

计算机视觉（CV）让计算机能够"看懂"图像和视频。应用包括人脸识别、物体检测、医学图像分析等。深度学习大幅提升了 CV 的性能。"""

    compare_chunking_methods(text)


demo_method_comparison()

示例 5: AI 切片 vs 固定切片对比

文本长度：370 字符

┌────────────────┬──────────┬─────────────┬──────────────┐
│    切片方法     │ 切片数量  │  平均长度    │   语义完整度  │
├────────────────┼──────────┼─────────────┼──────────────┤
│  固定字符切片   │    3     │  123.3   │     ⭐⭐        │
│  滑动窗口切片   │    5     │  133.0   │     ⭐⭐⭐       │
│  AI 辅助切片     │    3     │  122.0   │     ⭐⭐⭐⭐      │
└────────────────┴──────────┴─────────────┴──────────────┘

💡 说明：
   语义完整度是主观评估，AI 切片通常能保持更好的语义连贯性


## 示例 6: AI 切片最佳实践

In [14]:
def ai_chunking_best_practices():
    """
    AI 切片的最佳实践建议
    """
    print("=" * 60)
    print("示例 6: AI 切片最佳实践")
    print("=" * 60)

    print("""
┌─────────────────────────────────────────────────────────┐
│ AI 辅助切片最佳实践                                      │
├─────────────────────────────────────────────────────────┤
│                                                         │
│ 1. 选择合适的触发时机                                   │
│    - 文档预处理时一次性切片（推荐）                     │
│    - 按需动态切片（灵活但慢）                           │
│                                                         │
│ 2. 设计有效的 Prompt                                    │
│    - 明确告知 AI 切片的用途（检索/问答）                │
│    - 指定期望的切片数量和大小                           │
│    - 要求返回结构化格式（JSON）                         │
│                                                         │
│ 3. 降低成本的方法                                        │
│    - 先用规则粗切，再用 AI 精切                          │
│    - 对长文档分段处理                                   │
│    - 使用本地模型（Ollama）                             │
│                                                         │
│ 4. 质量保证                                              │
│    - 检查切片是否为空                                   │
│    - 验证切片大小是否合理                               │
│    - 人工抽检切片质量                                   │
│                                                         │
└─────────────────────────────────────────────────────────┘

💡 推荐流程:

```
原始文档
    ↓
[预处理] 清理格式、统一编码
    ↓
[粗切] 按段落/章节分割（规则方法）
    ↓
[精切] 对每个粗切片段用 AI 识别语义边界
    ↓
[后处理] 合并过小切片，过滤空切片
    ↓
最终切片列表
```

📊 成本对比:

| 方法 | 速度 | 成本 | 质量 | 推荐场景 |
|------|------|------|------|----------|
| 固定切片 | 快 | 无 | ⭐⭐ | 原型/测试 |
| 滑动窗口 | 中 | 无 | ⭐⭐⭐ | 生产环境 |
| AI 切片 | 慢 | 有 | ⭐⭐⭐⭐ | 高质量 RAG |
| 混合方法 | 中 | 低 | ⭐⭐⭐⭐ | 推荐 |
""")


ai_chunking_best_practices()

示例 6: AI 切片最佳实践

┌─────────────────────────────────────────────────────────┐
│ AI 辅助切片最佳实践                                      │
├─────────────────────────────────────────────────────────┤
│                                                         │
│ 1. 选择合适的触发时机                                   │
│    - 文档预处理时一次性切片（推荐）                     │
│    - 按需动态切片（灵活但慢）                           │
│                                                         │
│ 2. 设计有效的 Prompt                                    │
│    - 明确告知 AI 切片的用途（检索/问答）                │
│    - 指定期望的切片数量和大小                           │
│    - 要求返回结构化格式（JSON）                         │
│                                                         │
│ 3. 降低成本的方法                                        │
│    - 先用规则粗切，再用 AI 精切                          │
│    - 对长文档分段处理                                   │
│    - 使用本地模型（Ollama）                             │
│                                                         │
│ 4. 质量保证            

---

## 总结

AI 辅助切片是一种利用大语言模型识别语义边界的智能切片方法。

**主要特点**：
- 切片质量最高，语义完整
- 需要调用 LLM API，有成本
- 适合高质量 RAG 应用

**推荐方法**：混合使用规则切片 + AI 精切，在保证质量的同时控制成本。

**下一步学习**：04_summary_chunking.py（概要生成切片）